# Preprocessing Dataset — Prediksi Ketepatan Lulus Mahasiswa

Notebook ini mendokumentasikan seluruh pipeline preprocessing untuk proyek prediksi ketepatan lulus mahasiswa. Dataset mentah (`dataset.csv`) berisi 608 baris dan 27 kolom dengan berbagai masalah kualitas data: missing values sistematik, system zeros pada nilai IPS, fitur leakage, dan korelasi redundan.

Pipeline ini mengubah dataset mentah menjadi `dataset_clean.csv` yang siap digunakan untuk pemodelan Decision Tree.


## 0. Setup & Load

Kita mulai dengan mengimpor library yang diperlukan, mengatur opsi tampilan pandas, dan memuat dataset mentah.


In [1]:
import pandas as pd
import numpy as np

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except:
    pass
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.read_csv('dataset.csv')
print(f'Shape: {df.shape}')
df.info()
print()
df.head()


Shape: (608, 27)
<class 'pandas.DataFrame'>
RangeIndex: 608 entries, 0 to 607
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   student_id            608 non-null    str    
 1   angkatan              608 non-null    int64  
 2   program               608 non-null    str    
 3   jenis_kelamin         608 non-null    str    
 4   id_agama              608 non-null    int64  
 5   ips_sem1              441 non-null    float64
 6   ips_sem2              489 non-null    float64
 7   ips_sem3              590 non-null    float64
 8   ips_sem4              417 non-null    float64
 9   ipk_sem4              608 non-null    float64
 10  sks_sem1              441 non-null    float64
 11  sks_sem2              489 non-null    float64
 12  sks_sem3              590 non-null    float64
 13  sks_sem4              417 non-null    float64
 14  total_sks_lulus_sem4  608 non-null    int64  
 15  failed_courses   

,student_id,angkatan,program,jenis_kelamin,id_agama,ips_sem1,ips_sem2,ips_sem3,ips_sem4,ipk_sem4,sks_sem1,sks_sem2,sks_sem3,sks_sem4,total_sks_lulus_sem4,failed_courses,failed_in_sem1,repeated_courses,avg_attendance,ips_trend,avg_ips,ips_std,ips_max,ips_min,sks_completion_ratio,semester_count,target
0,207421001,2020,IH,P,2,2.9000,3.4300,3.1700,3.6300,3.2800,20.0000,21.0000,18.0000,19.0000,78,1,1,0,83.6700,0.7300,3.2825,0.3170,3.6300,2.9000,0.9750,8,1
1,207421003,2020,IH,P,1,3.4500,3.4300,3.3300,3.2600,3.3700,20.0000,21.0000,18.0000,19.0000,78,1,0,0,81.6700,-0.1900,3.3675,0.0888,3.4500,3.2600,0.9750,8,1
2,207421004,2020,IH,L,1,3.1000,3.3300,3.3300,3.3700,3.2800,20.0000,21.0000,18.0000,19.0000,78,0,0,0,83.0800,0.2700,3.2825,0.1231,3.3700,3.1000,0.9750,8,1
3,207421005,2020,IH,L,1,3.1000,3.2400,3.6700,0.0000,2.5100,20.0000,21.0000,18.0000,19.0000,78,8,0,0,78.9700,-3.1000,2.5025,1.6859,3.6700,0.0000,0.9750,4,0
4,207421006,2020,IH,L,1,3.2000,3.4300,3.3300,3.6300,3.4000,20.0000,21.0000,18.0000,19.0000,78,0,0,0,82.0900,0.4300,3.3975,0.1814,3.6300,3.2000,0.9750,4,0


## 1. Replace System Zeros (IPS = 0.0 → NaN)

**Temuan EDA:** 36% mahasiswa (219 dari 608) memiliki minimal satu nilai IPS = 0.0, namun 79% di antaranya tetap lulus tepat waktu. Ini membuktikan bahwa nilai 0.0 bukanlah nilai riil, melainkan **placeholder dari sistem legacy**. Jika tidak diperbaiki, nilai ini akan merusak imputasi dan menghasilkan korelasi palsu (ips_sem1 r = -0.27 dengan target — negatif karena 0.0 dianggap rendah).

**Tindakan:** Ganti semua 0.0 dengan NaN pada kolom `ips_sem1-4` dan `sks_sem1-4`.


In [2]:
ips_cols = ['ips_sem1', 'ips_sem2', 'ips_sem3', 'ips_sem4']
sks_cols = ['sks_sem1', 'sks_sem2', 'sks_sem3', 'sks_sem4']

print('Jumlah nilai 0.0 sebelum replacement:')
for col in ips_cols + sks_cols:
    zeros = (df[col] == 0.0).sum()
    print(f'  {col}: {zeros} zeros')

for col in ips_cols + sks_cols:
    df.loc[df[col] == 0.0, col] = np.nan

print('\nJumlah nilai 0.0 setelah replacement:')
for col in ips_cols + sks_cols:
    zeros = (df[col] == 0.0).sum()
    print(f'  {col}: {zeros} zeros')
print('Semua system zero berhasil diganti NaN.')


Jumlah nilai 0.0 sebelum replacement:
  ips_sem1: 170 zeros
  ips_sem2: 2 zeros
  ips_sem3: 9 zeros
  ips_sem4: 42 zeros
  sks_sem1: 0 zeros
  sks_sem2: 0 zeros
  sks_sem3: 0 zeros
  sks_sem4: 0 zeros

Jumlah nilai 0.0 setelah replacement:
  ips_sem1: 0 zeros
  ips_sem2: 0 zeros
  ips_sem3: 0 zeros
  ips_sem4: 0 zeros
  sks_sem1: 0 zeros
  sks_sem2: 0 zeros
  sks_sem3: 0 zeros
  sks_sem4: 0 zeros
Semua system zero berhasil diganti NaN.


## 2. Feature Engineering: has_attendance

**Temuan EDA:** `avg_attendance` memiliki 52.8% missing value dan korelasi Pearson dengan target sebesar r = -0.016 — hampir nol. Nilai numerik kehadiran tidak mendiskriminasi antara mahasiswa yang lulus tepat waktu vs tidak. Namun, ada sinyal lemah pada **keberadaan** data kehadiran itu sendiri (selisih ~4.7% target rate antara yang punya data vs tidak).

**Tindakan:** Buat fitur biner `has_attendance` = 1 jika `avg_attendance` tidak null, 0 jika null. `avg_attendance` akan di-drop di langkah berikutnya.


In [3]:
df['has_attendance'] = df['avg_attendance'].notna().astype(int)

print('Distribusi has_attendance:')
print(df['has_attendance'].value_counts().to_dict())

present = df['has_attendance'].sum()
missing = len(df) - present
print(f'\nMahasiswa dengan data kehadiran : {present} ({present/len(df)*100:.1f}%)')
print(f'Mahasiswa tanpa data kehadiran : {missing} ({missing/len(df)*100:.1f}%)')


Distribusi has_attendance:
{0: 321, 1: 287}

Mahasiswa dengan data kehadiran : 287 (47.2%)
Mahasiswa tanpa data kehadiran : 321 (52.8%)


## 3. Drop Redundant & Leakage Features

Berdasarkan analisis korelasi dan domain knowledge, beberapa fitur harus di-drop:

| Fitur | Alasan | r dengan Target |
|-------|--------|:---:|
| `student_id` | Identitas unik, bukan prediktor | — |
| `ips_sem4` | **Data leakage** — mahasiswa lulus tepat waktu otomatis punya IPS sem4 tinggi | **+0.877** |
| `sks_sem4` | Terkait ips_sem4, leakage | -0.356 |
| `ipk_sem4` | Redundan dengan avg_ips (inter-korelasi > 0.8) | +0.633 |
| `semester_count` | **Circular** — target didefinisikan dari durasi studi | — |
| `ips_max` | Tidak mendiskriminasi kedua kelas (mean ≈ 3.65 untuk kedua kelas) | ≈ 0.000 |
| `total_sks_lulus_sem4` | Redundan dengan `sks_completion_ratio` | — |
| `avg_attendance` | 53% missing + korelasi hampir nol, sudah diganti `has_attendance` | -0.016 |


In [4]:
drop_cols = [
    'student_id',
    'ips_sem4',
    'sks_sem4',
    'ipk_sem4',
    'semester_count',
    'ips_max',
    'total_sks_lulus_sem4',
    'avg_attendance'
]

print('Drop kolom:')
for col in drop_cols:
    print(f'  - {col}')

df.drop(columns=drop_cols, inplace=True)
print(f'\nSisa kolom ({len(df.columns)}): {list(df.columns)}')


Drop kolom:
  - student_id
  - ips_sem4
  - sks_sem4
  - ipk_sem4
  - semester_count
  - ips_max
  - total_sks_lulus_sem4
  - avg_attendance

Sisa kolom (20): ['angkatan', 'program', 'jenis_kelamin', 'id_agama', 'ips_sem1', 'ips_sem2', 'ips_sem3', 'sks_sem1', 'sks_sem2', 'sks_sem3', 'failed_courses', 'failed_in_sem1', 'repeated_courses', 'ips_trend', 'avg_ips', 'ips_std', 'ips_min', 'sks_completion_ratio', 'target', 'has_attendance']


## 4. Missing Value Check (Pre-Imputation)

Sebelum imputasi, mari kita lihat distribusi missing value yang tersisa.

**Temuan EDA:** Missing bersifat sistematik, bukan random. Angkatan tua lebih parah:
- Angkatan 2015: rata-rata 1.99 missing per mahasiswa
- Angkatan 2018: rata-rata 1.85 missing
- Angkatan 2022: hanya 0.03 missing

Ini mengonfirmasi bahwa imputasi harus dilakukan per angkatan, bukan secara global.


In [6]:
print('Missing values per kolom (pre-imputation):')
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

print('\nRata-rata missing per angkatan:')
ips_sks_cols = ['ips_sem1', 'ips_sem2', 'ips_sem3', 'sks_sem1', 'sks_sem2', 'sks_sem3']
missing_by_angkatan = df.groupby('angkatan')[ips_sks_cols].apply(
    lambda x: x.isnull().sum(axis=1).mean()
)
print(missing_by_angkatan.to_string())


Missing values per kolom (pre-imputation):
ips_sem1     337
ips_sem2     121
ips_sem3      27
sks_sem1     167
sks_sem2     119
sks_sem3      18
ips_trend    118
avg_ips      118
ips_std      118
ips_min      118

Rata-rata missing per angkatan:
angkatan
2015   2.6810
2016   0.9815
2017   1.1875
2018   3.1739
2019   1.7037
2020   0.0250
2021   0.0000
2022   0.9558
2023   0.0400


## 5. Median Imputation per Angkatan

**Mengapa median?** Median robust terhadap outlier, berbeda dengan mean yang bisa terdistorsi oleh nilai IPS ekstrem.

**Mengapa per angkatan?** Pola missing sangat berbeda antar kohort — angkatan tua kehilangan lebih banyak data semester awal karena sistem pencatatan yang belum terkomputerisasi.

**Catatan:** Data hanya 608 baris, sehingga metode kompleks seperti KNN Imputer atau MICE bersifat overkill. Imputasi median per grup sudah memadai untuk Decision Tree.


In [7]:
impute_cols = ['ips_sem1', 'ips_sem2', 'ips_sem3', 'sks_sem1', 'sks_sem2', 'sks_sem3']

print('Imputasi per kolom:')
for col in impute_cols:
    n_before = df[col].isnull().sum()
    df[col] = df.groupby('angkatan')[col].transform(lambda x: x.fillna(x.median()))
    n_after = df[col].isnull().sum()
    print(f'  {col}: {n_before} nulls -> diimputasi -> {n_after} nulls tersisa')

remaining = df[impute_cols].isnull().sum().sum()
print(f'\nTotal nulls tersisa di kolom imputasi: {remaining}')

# Final null check
total_nulls = df.isnull().sum().sum()
print(f'\nVerifikasi final semua nulls: {total_nulls}')
if total_nulls > 0:
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print('  Tidak ada nulls tersisa!')


Imputasi per kolom:
  ips_sem1: 337 nulls -> diimputasi -> 0 nulls tersisa
  ips_sem2: 121 nulls -> diimputasi -> 0 nulls tersisa
  ips_sem3: 27 nulls -> diimputasi -> 0 nulls tersisa
  sks_sem1: 167 nulls -> diimputasi -> 0 nulls tersisa
  sks_sem2: 119 nulls -> diimputasi -> 0 nulls tersisa
  sks_sem3: 18 nulls -> diimputasi -> 0 nulls tersisa

Total nulls tersisa di kolom imputasi: 0

Verifikasi final semua nulls: 472
ips_trend    118
avg_ips      118
ips_std      118
ips_min      118
dtype: int64


## 6. Recompute Derived Features

Setelah imputasi, semua derived features (yang sebelumnya bergantung pada nilai IPS/SKS) harus dihitung ulang dari nilai yang sudah bersih:

| Fitur | Formula | Deskripsi |
|-------|---------|-----------|
| `ips_trend` | `ips_sem3 - ips_sem1` | Tren akademik (positif = membaik) |
| `avg_ips` | `mean(ips_sem1, ips_sem2, ips_sem3)` | Rata-rata semester 1-3 |
| `ips_std` | `std(ips_sem1, ips_sem2, ips_sem3)` | Volatilitas kinerja |
| `ips_min` | `min(ips_sem1, ips_sem2, ips_sem3)` | Nilai IPS terendah |
| `sks_completion_ratio` | `(sks_sem1 + sks_sem2 + sks_sem3) / 60` | Proporsi SKS terhadap target 60 SKS |


In [8]:
# ips_trend: last valid minus first valid
ips_have = df[['ips_sem1', 'ips_sem2', 'ips_sem3']]
df['ips_trend'] = ips_have.bfill(axis=1).iloc[:, -1] - ips_have.ffill(axis=1).iloc[:, 0]

# avg_ips
df['avg_ips'] = df[['ips_sem1', 'ips_sem2', 'ips_sem3']].mean(axis=1)

# ips_std
df['ips_std'] = df[['ips_sem1', 'ips_sem2', 'ips_sem3']].std(axis=1)

# ips_min
df['ips_min'] = df[['ips_sem1', 'ips_sem2', 'ips_sem3']].min(axis=1)

# sks_completion_ratio
df['sks_completion_ratio'] = (df['sks_sem1'] + df['sks_sem2'] + df['sks_sem3']) / 60

print('Statistik derived features:')
print(f'  ips_trend           : min={df.ips_trend.min():.4f}, max={df.ips_trend.max():.4f}, mean={df.ips_trend.mean():.4f}')
print(f'  avg_ips             : min={df.avg_ips.min():.4f}, max={df.avg_ips.max():.4f}, mean={df.avg_ips.mean():.4f}')
print(f'  ips_std             : min={df.ips_std.min():.4f}, max={df.ips_std.max():.4f}, mean={df.ips_std.mean():.4f}')
print(f'  ips_min             : min={df.ips_min.min():.4f}, max={df.ips_min.max():.4f}, mean={df.ips_min.mean():.4f}')
print(f'  sks_completion_ratio: min={df.sks_completion_ratio.min():.4f}, max={df.sks_completion_ratio.max():.4f}, mean={df.sks_completion_ratio.mean():.4f}')


Statistik derived features:
  ips_trend           : min=-2.7900, max=2.5000, mean=0.3768
  avg_ips             : min=1.6800, max=3.8967, mean=3.2993
  ips_std             : min=0.0000, max=1.8036, mean=0.3013
  ips_min             : min=0.3000, max=3.8000, mean=3.0458
  sks_completion_ratio: min=0.5000, max=2.4833, mean=1.3606


## 7. Encode Categorical Features

**Temuan EDA:** Program IH hampir 2x lebih berisiko tidak tepat waktu dibanding AP (12.6% vs 6.8%). Decision Tree di sklearn membutuhkan input numerik, jadi semua fitur kategorikal harus di-encode.

**Encoding:**
- `program`: AP → 0, IH → 1
- `jenis_kelamin`: L → 0, P → 1
- `id_agama`: Biarkan apa adanya (sudah numerik: 1, 2, 4)


In [9]:
df['program'] = df['program'].map({'AP': 0, 'IH': 1})
df['jenis_kelamin'] = df['jenis_kelamin'].map({'L': 0, 'P': 1})

print('Unique values setelah encoding:')
print(f'  program       : {sorted(df.program.unique())}')
print(f'  jenis_kelamin : {sorted(df.jenis_kelamin.unique())}')
print(f'  id_agama      : {sorted(df.id_agama.unique())}')


Unique values setelah encoding:
  program       : [np.int64(0), np.int64(1)]
  jenis_kelamin : [np.int64(0), np.int64(1)]
  id_agama      : [np.int64(1), np.int64(2), np.int64(4)]


## 8. Final Dataset Summary

Setelah seluruh pipeline preprocessing selesai, mari kita reorder kolom, lalu verifikasi dataset final sebelum disimpan.


In [10]:
# Reorder columns to match expected output
expected_order = [
    'angkatan', 'program', 'jenis_kelamin', 'id_agama',
    'ips_sem1', 'ips_sem2', 'ips_sem3',
    'sks_sem1', 'sks_sem2', 'sks_sem3',
    'failed_courses', 'failed_in_sem1', 'repeated_courses',
    'has_attendance',
    'ips_trend', 'avg_ips', 'ips_std', 'ips_min', 'sks_completion_ratio',
    'target'
]
actual_order = [c for c in expected_order if c in df.columns]
df = df[actual_order]

print(f'Final shape: {df.shape}')
print(f'Total NULLs: {df.isnull().sum().sum()}')
print(f'\nInfo:')
df.info()
print(f'\nDeskripsi statistik:')
df.describe()
print(f'\nDistribusi target:')
target_counts = df.target.value_counts()
target_pct = df.target.mean() * 100
print(f'  {target_counts.to_dict()}')
print(f'  Tepat waktu: {target_pct:.2f}%')
print(f'  Tidak tepat: {100 - target_pct:.2f}%')
print(f'\n5 baris pertama:')
df.head()


Final shape: (608, 20)
Total NULLs: 0

Info:


<class 'pandas.DataFrame'>
RangeIndex: 608 entries, 0 to 607
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   angkatan              608 non-null    int64  
 1   program               608 non-null    int64  
 2   jenis_kelamin         608 non-null    int64  
 3   id_agama              608 non-null    int64  
 4   ips_sem1              608 non-null    float64
 5   ips_sem2              608 non-null    float64
 6   ips_sem3              608 non-null    float64
 7   sks_sem1              608 non-null    float64
 8   sks_sem2              608 non-null    float64
 9   sks_sem3              608 non-null    float64
 10  failed_courses        608 non-null    int64  
 11  failed_in_sem1        608 non-null    int64  
 12  repeated_courses      608 non-null    int64  
 13  has_attendance        608 non-null    int64  
 14  ips_trend             608 non-null    float64
 15  avg_ips               608 non-null

,angkatan,program,jenis_kelamin,id_agama,ips_sem1,ips_sem2,ips_sem3,sks_sem1,sks_sem2,sks_sem3,failed_courses,failed_in_sem1,repeated_courses,has_attendance,ips_trend,avg_ips,ips_std,ips_min,sks_completion_ratio,target
0,2020,1,1,2,2.9000,3.4300,3.1700,20.0000,21.0000,18.0000,1,1,0,1,0.2700,3.1667,0.2650,2.9000,0.9833,1
1,2020,1,1,1,3.4500,3.4300,3.3300,20.0000,21.0000,18.0000,1,0,0,1,-0.1200,3.4033,0.0643,3.3300,0.9833,1
2,2020,1,0,1,3.1000,3.3300,3.3300,20.0000,21.0000,18.0000,0,0,0,1,0.2300,3.2533,0.1328,3.1000,0.9833,1
3,2020,1,0,1,3.1000,3.2400,3.6700,20.0000,21.0000,18.0000,8,0,0,1,0.5700,3.3367,0.2970,3.1000,0.9833,0
4,2020,1,0,1,3.2000,3.4300,3.3300,20.0000,21.0000,18.0000,0,0,0,1,0.1300,3.3200,0.1153,3.2000,0.9833,0


## 9. Save Clean Dataset

Simpan dataset yang sudah bersih ke `dataset_clean.csv`.


In [11]:
df.to_csv('dataset_clean_nb.csv', index=False)
print(f'Dataset berhasil disimpan: dataset_clean.csv ({df.shape[0]} rows x {df.shape[1]} cols)')
print('Pipeline preprocessing selesai.')


Dataset berhasil disimpan: dataset_clean.csv (608 rows x 20 cols)
Pipeline preprocessing selesai.
